In [1]:
import pandas as pd, numpy as np
import xgboost as xgb
import mlflow
from sklearn.metrics import average_precision_score, roc_auc_score

f = pd.read_parquet("data/features.parquet")
raw = pd.read_csv("data/train_transaction.csv",
                  usecols=["TransactionID", "TransactionAmt", "ProductCD", "card4", "card6"])
df = f.merge(raw, on="TransactionID", how="left")
df = df.sort_values("event_timestamp").reset_index(drop=True)

for c in ["ProductCD", "card4", "card6"]:
    df[c] = df[c].astype("category")

features = ["hour", "day_of_week", "tx_count_before", "secs_since_last", "amt_vs_avg",
            "device_changed", "addr_changed", "email_changed",
            "TransactionAmt", "ProductCD", "card4", "card6"]

cut = int(len(df) * 0.8)
train, test = df.iloc[:cut], df.iloc[cut:]
X_train, y_train = train[features], train["isFraud"]
X_test, y_test = test[features], test["isFraud"]

ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("train rows:", len(train), "test rows:", len(test))
print("fraud rate train:", y_train.mean(), "test:", y_test.mean())
print("class weight (scale_pos_weight):", round(ratio, 1))

train rows: 472432 test rows: 118108
fraud rate train: 0.03513521522674162 test: 0.034409184813899145
class weight (scale_pos_weight): 27.5


In [2]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fraud-detection")

params = dict(n_estimators=300, max_depth=6, learning_rate=0.1,
              subsample=0.8, colsample_bytree=0.8,
              scale_pos_weight=ratio, tree_method="hist",
              enable_categorical=True, eval_metric="aucpr", random_state=42)

with mlflow.start_run(run_name="xgb_baseline"):
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

    proba = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, proba)
    roc = roc_auc_score(y_test, proba)

    mlflow.log_params({k: v for k, v in params.items() if k != "enable_categorical"})
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc", roc)
    mlflow.xgboost.log_model(model, artifact_path="model")

print("PR-AUC:", round(pr_auc, 4), " ROC-AUC:", round(roc, 4))
print("Baseline PR-AUC if guessing randomly:", round(y_test.mean(), 4))

[0]	validation_0-aucpr:0.11126
[50]	validation_0-aucpr:0.12853
[100]	validation_0-aucpr:0.12390
[150]	validation_0-aucpr:0.12174
[200]	validation_0-aucpr:0.11663
[250]	validation_0-aucpr:0.11494
[299]	validation_0-aucpr:0.11852


2026/10/24 19:05:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


PR-AUC: 0.1187  ROC-AUC: 0.7535
Baseline PR-AUC if guessing randomly: 0.0344


In [ ]:
import os
os.makedirs("models", exist_ok=True)
model.save_model("models/xgb.json")
print("saved")